# Laboratorul 1 - Procesare Big Data

## Etapa 1: Alegerea dataset-ului

În această etapă vom analiza datasetul ales pentru procesare big data.

In [3]:
# Colab's dataproc-spark-connect requires PySpark 4.0.x; Delta Lake 4.0.0 supports Spark 4.0.
%pip install -q 'pyspark[connect]==4.0.0' delta-spark==4.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 1.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 14.7 MB/s eta 0:00:00


In [4]:
# Import librării necesare
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

# Configurare vizualizări
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Proiectul și toate ieșirile sunt păstrate în Google Drive.
PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/PBC-Lab-1'
DATA_ROOT = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_ROOT, 'raw')
os.makedirs(RAW_DIR, exist_ok=True)

dataset_name = 'Crimes_-_2001_to_Present_20260925.csv'
file_path = os.path.join(RAW_DIR, dataset_name)
legacy_candidates = [
    '/content/drive/MyDrive/Colab Notebooks/' + dataset_name,
    '/content/drive/My Drive/Colab Notebooks/' + dataset_name,
]
if not os.path.isfile(file_path):
    legacy_path = next((path for path in legacy_candidates if os.path.isfile(path)), None)
    if legacy_path is None:
        raise FileNotFoundError(
            f'Nu am găsit {dataset_name}. Încărcați CSV-ul în {RAW_DIR} '
            'sau păstrați-l în Colab Notebooks pentru copierea inițială.'
        )
    import shutil
    shutil.copy2(legacy_path, file_path)

file_size = os.path.getsize(file_path)
print(f'Fișier sursă: {file_path}')
print(f'Dimensiune fișier: {file_size / (1024 ** 3):.2f} GB')

Fișier sursă: /content/drive/MyDrive/Colab Notebooks/PBC-Lab-1/data/raw/Crimes_-_2001_to_Present_20260925.csv
Dimensiune fișier: 2.24 GB


In [7]:
# Citire primele 10 rânduri pentru a înțelege structura
try:
    df_sample = pd.read_csv(file_path, nrows=10)
    print("Coloanele din dataset:")
    print(df_sample.columns.tolist())
    print("\nPrimele 10 rânduri:")
    print(df_sample)
    print("\nTipuri de date:")
    print(df_sample.dtypes)
except Exception as e:
    print(f"Eroare la citirea fișierului: {e}")

Coloanele din dataset:
['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']

Primele 10 rânduri:
         ID Case Number                    Date  \
0  14329936    JK415994  09/16/2026 12:00:00 AM   
1  14332840    JK419513  09/16/2026 12:00:00 AM   
2  14329074    JK414859  09/16/2026 12:00:00 AM   
3  14329468    JK415287  09/16/2026 12:00:00 AM   
4  14330539    JK416884  09/16/2026 12:00:00 AM   
5  14329575    JK415610  09/16/2026 12:00:00 AM   
6  14336390    JK423947  09/16/2026 12:00:00 AM   
7  14329284    JK415110  09/16/2026 12:00:00 AM   
8  14333202    JK420095  09/16/2026 12:00:00 AM   
9  14330639    JK416997  09/16/2026 12:00:00 AM   

                                 Block  IUCR         Primary Type  \
0                  048XX N KIMBALL AVE   486        

## Descrierea dataset-ului

**Link către datasetul original:** [Crimes - 2001 to Present | City of Chicago | Data Portal](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data)

Datasetul conține informații despre infracțiunile raportate în Chicago de la 2001 până în prezent. Acesta include detalii precum data și ora infracțiunii, tipul de crimă, locația, districtul de poliție și alte informații relevante.

Am ales acest dataset pentru că:
1. Are dimensiunea corespunzătoare cerințelor (peste 2GB)
2. Conține date temporale (timestamp-uri) utile pentru analize ulterioare
3. Este un exemplu realist de date de tip Big Data
4. Oferă oportunități de analiză și procesare complexă

## Etapa 2: Ingestie și stocare

În această etapă inițializăm Spark cu Delta Lake și încărcăm CSV-ul cu o schemă declarată explicit. Comparația formatelor și scrierile de stocare sunt realizate separat în etapa dedicată.

In [8]:
# Inițializare Spark cu extensiile necesare pentru Delta Lake.
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (
    SparkSession.builder
    .appName('Chicago_Crimes_Analysis')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f'Sesiune PySpark inițializată: Spark {spark.version}')

Sesiune PySpark inițializată: Spark 4.0.0


In [9]:
# Schema explicită pentru CSV; identificatorii și codurile rămân șiruri când zerourile inițiale contează.
crime_schema = StructType([
    StructField('ID', LongType(), True),
    StructField('Case Number', StringType(), True),
    StructField('Date', StringType(), True),
    StructField('Block', StringType(), True),
    StructField('IUCR', StringType(), True),
    StructField('Primary Type', StringType(), True),
    StructField('Description', StringType(), True),
    StructField('Location Description', StringType(), True),
    StructField('Arrest', BooleanType(), True),
    StructField('Domestic', BooleanType(), True),
    StructField('Beat', StringType(), True),
    StructField('District', StringType(), True),
    StructField('Ward', IntegerType(), True),
    StructField('Community Area', IntegerType(), True),
    StructField('FBI Code', StringType(), True),
    StructField('X Coordinate', DoubleType(), True),
    StructField('Y Coordinate', DoubleType(), True),
    StructField('Year', IntegerType(), True),
    StructField('Updated On', StringType(), True),
    StructField('Latitude', StringType(), True),
    StructField('Longitude', StringType(), True),
    StructField('Location', StringType(), True),
])

def read_source_csv(path):
    raw_df = (
        spark.read.schema(crime_schema)
        .option('header', 'true')
        .option('mode', 'FAILFAST')
        .csv(path)
    )
    return (
        raw_df
        .withColumn('Date', to_timestamp(col('Date'), 'MM/dd/yyyy hh:mm:ss a'))
        .withColumn('Updated On', to_timestamp(col('Updated On'), 'MM/dd/yyyy hh:mm:ss a'))
        .withColumn('Latitude', regexp_replace(col('Latitude'), ',', '.').cast(DoubleType()))
        .withColumn('Longitude', regexp_replace(col('Longitude'), ',', '.').cast(DoubleType()))
    )

crimes_df = read_source_csv(file_path)
print(f'Număr total de înregistrări: {crimes_df.count()}')
print(f'Număr coloane: {len(crimes_df.columns)}')
crimes_df.printSchema()

Număr total de înregistrări: 8645530
Număr coloane: 22
root
 |-- ID: long (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [10]:
# Afișare primele 5 rânduri
crimes_df.show(5)

+--------+-----------+-------------------+-------------------+----+------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+
|      ID|Case Number|               Date|              Block|IUCR|Primary Type|         Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|         Updated On|    Latitude|    Longitude|            Location|
+--------+-----------+-------------------+-------------------+----+------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------+
|14329936|   JK415994|2026-09-16 00:00:00|048XX N KIMBALL AVE|0486|     BATTERY|DOMESTIC BATTERY ...|               ALLEY| false|    true|1712|     017|  33|

In [11]:
# Scrierea Delta partiționată și testele de versiuni sunt grupate în Etapa 4.
print('Datele sursă sunt încărcate; formatele de stocare vor fi create în Etapa 4.')

Datele sursă sunt încărcate; formatele de stocare vor fi create în Etapa 4.


In [12]:
# Păstrăm DataFrame-ul sursă nemodificat pentru curățare și comparații.
crimes_df.show(5, truncate=False)

+--------+-----------+-------------------+-------------------+----+------------+---------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date               |Block              |IUCR|Primary Type|Description                |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On         |Latitude    |Longitude    |Location                     |
+--------+-----------+-------------------+-------------------+----+------------+---------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+-----------------------------+
|14329936|JK415994   |2026-09-16 00:00:00|048XX N KIMBALL AVE|0486|BATTERY     |DOMESTIC BATTERY SIMPLE    |A

### Beneficiile utilizării Delta Lake față de Parquet simplu:

1. **ACID Transactions**: Asigură consistența datelor prin tranzacții ACID, permițând operatii concurente fără coruperea datelor.
2. **Schema Evolution**: Permite adăugarea sau modificarea coloanelor fără recrearea întregii baze de date.
3. **Time Travel**: Posibilitatea de a accesa versiuni anterioare ale datelor pentru audit sau recuperare.
4. **Optimizare automată**: Compacțare și clustering automat pentru performanță crescută.
5. **Upserts și deletes**: Suport nativ pentru operații de tip upsert (merge) și ștergere.
6. **Audit trail**: Jurnal complet al tuturor modificărilor efectuate asupra datelor.
7. **Compatibilitate cu Apache Spark**: Integrare perfectă cu ecosistemul Spark.

## Etapa 3: Curățare și transformare

În această etapă vom curăța datele, vom converti tipurile de date și vom face agregări simple.

In [13]:
# Verificare valori lipsă
print('Numărul valorilor nule sau goale pe coloană:')
null_counts = crimes_df.select([
    sum(when(col(column_name).isNull() | (col(column_name).cast('string') == ''), 1).otherwise(0))
    .alias(column_name)
    for column_name in crimes_df.columns
])
null_counts.show(truncate=False)

Numărul valorilor nule sau goale pe coloană:
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|ID |Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|Ward  |Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|0  |0          |0   |0    |0   |0           |0          |16699               |0     |0       |0   |47      |614811|613722        |0       |100091      |100091      |0   |0         |100091  |100091   |100091  |
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+--

In [14]:
# Alternativă pentru verificarea valorilor lipsă
null_counts = crimes_df.select([count(when(col(c).isNull(), c)).alias(c) for c in crimes_df.columns])
null_counts.show()

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
| ID|Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|  Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|  0|          0|   0|    0|   0|           0|          0|               16699|     0|       0|   0|      47|614811|        613722|       0|      100091|      100091|   0|         0|  100091|   100091|  100091|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

In [15]:
# Conversia tipurilor de date - în special pentru coloanele de timp
# Vom presupune că există o coloană de tip dată (de exemplu, 'Date')
# Vom verifica coloanele care ar putea conține date/timp

# Afișare statistici descriptive pentru coloanele numerice
numeric_cols = []
for field in crimes_df.schema.fields:
    if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType)):
        numeric_cols.append(field.name)

if numeric_cols:
    print("Statistici descriptive pentru coloanele numerice:")
    crimes_df.select(numeric_cols).describe().show()
else:
    print("Nu s-au găsit coloane numerice în dataset.")

Statistici descriptive pentru coloanele numerice:
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+
|summary|                ID|              Ward|    Community Area|      X Coordinate|     Y Coordinate|              Year|           Latitude|          Longitude|
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+
|  count|           8645530|           8030719|           8031808|           8545439|          8545439|           8645530|            8545439|            8545439|
|   mean|  7686308.09626998|22.782345640533556|37.337577666199195|1164683.2121564497|1885942.456667703|2011.3826918650448|  41.84261969788857| -87.67118694265166|
| stddev|3877870.8395691956|13.863658253850558|21.550514301820257|    16934.93758757|32402.49552138572| 7.359130058828035|0.08914913524

In [16]:
# Date și Updated On sunt deja convertite la timestamp la ingestia cu schemă explicită.
date_fields = [field for field in crimes_df.schema.fields if 'date' in field.name.lower() or 'time' in field.name.lower()]
date_cols = [field.name for field in date_fields]
print(f"Coloane posibile de tip dată: {date_cols}")

string_date_fields = [field.name for field in date_fields if isinstance(field.dataType, StringType)]
if string_date_fields:
    for col_name in string_date_fields:
        crimes_df = crimes_df.withColumn(col_name, to_timestamp(col(col_name), 'MM/dd/yyyy hh:mm:ss a'))
        print(f"Coloana {col_name} a fost convertită la tip timestamp.")
else:
    print('Coloanele temporale au fost deja normalizate în timpul ingestiei.')

Coloane posibile de tip dată: ['Date', 'Updated On']
Coloanele temporale au fost deja normalizate în timpul ingestiei.


In [17]:
# Curățare date - eliminare valori lipsă și duplicate
initial_count = crimes_df.count()
print(f"Numărul inițial de înregistrări: {initial_count}")

# Eliminare duplicate
crimes_df_clean = crimes_df.dropDuplicates()
after_dedupe_count = crimes_df_clean.count()
print(f"Numărul de înregistrări după eliminarea duplicatelor: {after_dedupe_count}")
print(f"Duplicate eliminate: {initial_count - after_dedupe_count}")

# Eliminare rânduri cu valori esențiale lipsă (presupunând că ID-ul incidentului este esențial)
essential_cols = [field.name for field in crimes_df.schema.fields if 'id' in field.name.lower()]
if essential_cols:
    crimes_df_clean = crimes_df_clean.na.drop(subset=essential_cols)
    after_na_drop_count = crimes_df_clean.count()
    print(f"Numărul de înregistrări după eliminarea valorilor lipsă pentru coloanele esențiale: {after_na_drop_count}")
    print(f"Înregistrări eliminate datorită valorilor lipsă: {after_dedupe_count - after_na_drop_count}")
else:
    crimes_df_clean = crimes_df_clean.na.drop()
    after_na_drop_count = crimes_df_clean.count()
    print(f"Numărul de înregistrări după eliminarea tuturor valorilor lipsă: {after_na_drop_count}")
    print(f"Înregistrări eliminate datorită valorilor lipsă: {after_dedupe_count - after_na_drop_count}")

Numărul inițial de înregistrări: 8645530
Numărul de înregistrări după eliminarea duplicatelor: 8645530
Duplicate eliminate: 0
Numărul de înregistrări după eliminarea valorilor lipsă pentru coloanele esențiale: 8645530
Înregistrări eliminate datorită valorilor lipsă: 0


In [18]:
# Agregări simple
print("Agregări simple:")

# Numărul de crime pe an
if date_cols:
    crimes_by_year = crimes_df_clean.withColumn("year", year(col(date_cols[0]))) \
                                    .groupBy("year") \
                                    .count() \
                                    .orderBy("year")
    print("Numărul de crime pe ani:")
    crimes_by_year.show(10)
else:
    print("Nu s-au putut face agregări pe ani deoarece nu s-a identificat o coloană de tip dată.")

# Tipuri de crime frecvente
crime_types = [col_name for col_name in crimes_df_clean.columns if 'primary' in col_name.lower() or 'type' in col_name.lower() or 'crime' in col_name.lower()]
if crime_types:
    print(f"\nTipuri de crime frecvente (bazat pe coloana {crime_types[0]}):")
    crimes_df_clean.groupBy(crime_types[0]).count().orderBy(desc("count")).show(10)
else:
    print("\nNu s-au putut identifica coloane referitoare la tipul de crimă.")

# Crime pe district
district_cols = [col_name for col_name in crimes_df_clean.columns if 'district' in col_name.lower()]
if district_cols:
    print(f"\nCrime pe district (bazat pe coloana {district_cols[0]}):")
    crimes_df_clean.groupBy(district_cols[0]).count().orderBy(desc("count")).show(10)
else:
    print("\nNu s-au putut identifica coloane referitoare la district.")

Agregări simple:
Numărul de crime pe ani:
+----+------+
|year| count|
+----+------+
|2001|485977|
|2002|486842|
|2003|476013|
|2004|469447|
|2005|453803|
|2006|448211|
|2007|437117|
|2008|427231|
|2009|392876|
|2010|370588|
+----+------+
only showing top 10 rows

Tipuri de crime frecvente (bazat pe coloana Primary Type):
+-------------------+-------+
|       Primary Type|  count|
+-------------------+-------+
|              THEFT|1836021|
|            BATTERY|1574678|
|    CRIMINAL DAMAGE| 981919|
|          NARCOTICS| 769814|
|            ASSAULT| 583830|
|      OTHER OFFENSE| 540207|
|           BURGLARY| 457757|
|MOTOR VEHICLE THEFT| 447822|
| DECEPTIVE PRACTICE| 402712|
|            ROBBERY| 319083|
+-------------------+-------+
only showing top 10 rows

Crime pe district (bazat pe coloana District):
+--------+------+
|District| count|
+--------+------+
|     008|578660|
|     011|546237|
|     006|504637|
|     004|488962|
|     025|486843|
|     007|486185|
|     003|439667|
|   

In [19]:
# Joins - exemplu ipotetic (dacă am avea un alt tabel cu informații suplimentare)
# Vom crea un tabel ipotetic pentru exemplificare
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Presupunem că avem un tabel cu informații despre district
district_info_schema = StructType([
    StructField("District", StringType(), True),
    StructField("Population", IntegerType(), True),
    StructField("Area_sq_mi", DoubleType(), True)
])

district_info_data = [
    ('001', 50000, 1.2),
    ('002', 75000, 2.1),
    ('003', 60000, 1.8),
    ('004', 80000, 2.5),
    ('005', 70000, 2.0)
]

district_info_df = spark.createDataFrame(district_info_data, district_info_schema)
print("Tabelul ipotetic cu informații despre district:")
district_info_df.show()

# Căutăm coloana District în datele originale
district_col = [col_name for col_name in crimes_df_clean.columns if 'district' in col_name.lower()]
if district_col:
    joined_df = crimes_df_clean.join(district_info_df, crimes_df_clean[district_col[0]] == district_info_df["District"], "inner")
    print(f"\nJoin între datele de crime și informațiile despre district (pe baza coloanei {district_col[0]}):")
    print(f"Numărul de înregistrări după join: {joined_df.count()}")
    selected_columns = [crimes_df_clean[district_col[0]], 'Population', 'Area_sq_mi']
    if crime_types:
        selected_columns.append(crimes_df_clean[crime_types[0]])
    joined_df.select(*selected_columns).show(5)
else:
    print("\nNu s-a putut efectua join deoarece nu s-a găsit coloana District.")

Tabelul ipotetic cu informații despre district:
+--------+----------+----------+
|District|Population|Area_sq_mi|
+--------+----------+----------+
|     001|     50000|       1.2|
|     002|     75000|       2.1|
|     003|     60000|       1.8|
|     004|     80000|       2.5|
|     005|     70000|       2.0|
+--------+----------+----------+


Join între datele de crime și informațiile despre district (pe baza coloanei District):
Numărul de înregistrări după join: 2084317
+--------+----------+----------+------------------+
|District|Population|Area_sq_mi|      Primary Type|
+--------+----------+----------+------------------+
|     001|     50000|       1.2|             THEFT|
|     001|     50000|       1.2|DECEPTIVE PRACTICE|
|     001|     50000|       1.2|             THEFT|
|     001|     50000|       1.2|             THEFT|
|     001|     50000|       1.2|           BATTERY|
+--------+----------+----------+------------------+
only showing top 5 rows


### Decizia privind procesarea datelor

Am ales să procesăm datele astfel:
1. **Eliminarea duplicatelor**: Pentru a evita distorsionarea statisticilor
2. **Eliminarea valorilor lipsă**: Am păstrat doar înregistrările cu informații esențiale complete
3. **Conversia tipurilor de date**: Am convertit coloanele de tip dată la formatul corespunzător pentru analiză
4. **Agregări simple**: Am calculat numărul de crime pe ani, tipuri frecvente de crime și distribuția pe district

Am decis ce date sunt eronate sau trebuie filtrate pe baza:
- Completitudinii informațiilor esențiale (cum ar fi ID-ul incidentului)
- Validității valorilor (verificarea dacă datele sunt într-un interval rezonabil)
- Formatului corect al datelor (de exemplu, formatul datelor și orele)

### Importanța verificării valorilor lipsă sau eronate

Este esențial să verificăm valorile lipsă sau eronate înainte de a calcula metricile numerice pentru că:
1. **Distorsionează statisticile**: Valorile lipsă sau eronate pot duce la calcule incorecte și concluzii greșite
2. **Afectează performanța**: Procesarea datelor necurate poate fi ineficientă și consumatoare de resurse
3. **Reduc acuratețea modelelor**: Dacă datele sunt folosite pentru modelare predictivă, datele necurate reduc semnificativ acuratețea
4. **Crește incertitudinea**: Datele de calitate slabă cresc incertitudinea în analizele statistice
5. **Pot cauza erori în execuție**: Unele funcții nu pot gestiona corect valorile lipsă sau eronate

## Etapa 4: CSV, Parquet, partiționare și Delta Lake

Comparația folosește DataFrame-ul sursă `crimes_df`, nu versiunea curățată din etapa precedentă. Interogarea identică filtrează anul cu cele mai multe înregistrări și agregă după `Primary Type`. Benchmarkurile fac trei acțiuni complete pentru fiecare format și pot dura; fișierele rămân în Google Drive pentru a fi disponibile la laborator.

In [20]:
from time import perf_counter
from statistics import median
import shutil

REPEATS = 3
PARQUET_PATH = os.path.join(DATA_ROOT, 'parquet')
PARTITIONED_PATH = os.path.join(DATA_ROOT, 'parquet_partitioned')
DELTA_PATH = os.path.join(DATA_ROOT, 'delta')
PARTITION_COLUMN = 'Year'

def measure(operation, repeats=REPEATS):
    elapsed = []
    for _ in range(repeats):
        start = perf_counter()
        operation()
        elapsed.append(perf_counter() - start)
    return elapsed, median(elapsed)

def directory_size_bytes(path):
    total = 0
    for root, _, filenames in os.walk(path):
        for filename in filenames:
            total += os.path.getsize(os.path.join(root, filename))
    return total

def size_in_mb(size_bytes):
    return size_bytes / (1024 ** 2)

### Parquet nepartiționat și partiționat

Directorul `data/raw` păstrează CSV-ul inițial. Scrierile Parquet folosesc `overwrite`, apoi recitim datele și verificăm numărul de rânduri și schema înainte de a continua.

In [21]:
SOURCE_COUNT = crimes_df.count()
SOURCE_SCHEMA = crimes_df.schema

crimes_df.write.mode('overwrite').parquet(PARQUET_PATH)
parquet_df = spark.read.parquet(PARQUET_PATH)
PARQUET_COUNT = parquet_df.count()
assert PARQUET_COUNT == SOURCE_COUNT, 'Numărul de rânduri Parquet diferă de sursă.'
assert parquet_df.schema == SOURCE_SCHEMA, 'Schema Parquet diferă de sursă.'
print(f'Parquet verificat: {PARQUET_COUNT:,} rânduri; schema identică.')

year_cardinality = crimes_df.select(PARTITION_COLUMN).distinct().count()
print(f'Valori distincte în {PARTITION_COLUMN}: {year_cardinality}')
if year_cardinality > 100:
    raise ValueError('Cardinalitate prea mare pentru partiționarea aleasă.')

crimes_df.write.mode('overwrite').partitionBy(PARTITION_COLUMN).parquet(PARTITIONED_PATH)
partitioned_df = spark.read.parquet(PARTITIONED_PATH).select(*crimes_df.columns)
assert partitioned_df.count() == SOURCE_COUNT, 'Numărul de rânduri partiționat diferă de sursă.'
assert partitioned_df.schema == SOURCE_SCHEMA, 'Schema partiționată diferă de sursă.'
print(f'Parquet partiționat verificat în {PARTITIONED_PATH}.')

Parquet verificat: 8,645,530 rânduri; schema identică.
Valori distincte în Year: 26
Parquet partiționat verificat în /content/drive/MyDrive/Colab Notebooks/PBC-Lab-1/data/parquet_partitioned.


### Tabel Delta Lake

Schema sursei conține nume cu spații (de exemplu `Case Number` și `Primary Type`). Activăm column mapping în modul `name`, care permite păstrarea acestor nume în Delta fără redenumirea coloanelor folosite de restul notebookului. Pentru ca versiunea inițială a experimentului să fie versiunea 0, celula următoare recreează numai directorul generat `data/delta`. Nu folosiți acest director pentru date Delta pe care doriți să le păstrați.

In [23]:
if os.path.exists(DELTA_PATH):
    shutil.rmtree(DELTA_PATH)

(
    crimes_df.write.format('delta')
    .option('delta.columnMapping.mode', 'name')
    .option('delta.minReaderVersion', '2')
    .option('delta.minWriterVersion', '5')
    .mode('overwrite')
    .partitionBy(PARTITION_COLUMN)
    .save(DELTA_PATH)
)
delta_df = spark.read.format('delta').load(DELTA_PATH)
DELTA_INITIAL_COUNT = delta_df.count()
assert DELTA_INITIAL_COUNT == SOURCE_COUNT, 'Numărul de rânduri Delta diferă de sursă.'
assert delta_df.schema == SOURCE_SCHEMA, 'Schema Delta diferă de sursă.'
print(f'Delta verificat: {DELTA_INITIAL_COUNT:,} rânduri; schema identică.')
print(f'Fișiere jurnal Delta: {len(os.listdir(os.path.join(DELTA_PATH, "_delta_log")))}')

Delta verificat: 8,645,530 rânduri; schema identică.
Fișiere jurnal Delta: 5


In [24]:
# Anul cel mai frecvent este prezent în date și este comun tuturor comparațiilor.
most_common_year = (
    crimes_df.where(col(PARTITION_COLUMN).isNotNull())
    .groupBy(PARTITION_COLUMN).count()
    .orderBy(desc('count'), col(PARTITION_COLUMN))
    .first()
)
if most_common_year is None:
    raise ValueError('Nu există valori de an ne-nule pentru filtrare.')
TEST_YEAR = int(most_common_year[PARTITION_COLUMN])
print(f'Anul de filtrare folosit în toate interogările: {TEST_YEAR}')

partition_directories = sorted(
    name for name in os.listdir(PARTITIONED_PATH)
    if name.startswith(f'{PARTITION_COLUMN}=')
)
print('Directoare Parquet partiționate (primele 20):')
print(partition_directories[:20])
print('Fiecare cale Year=valoare corespunde valorii stocate în coloana Year.')

unpartitioned_filter = parquet_df.filter(col(PARTITION_COLUMN) == TEST_YEAR)
partitioned_filter = partitioned_df.filter(col(PARTITION_COLUMN) == TEST_YEAR)
print('\nPlan Parquet nepartiționat:')
unpartitioned_filter.explain('formatted')
print('\nPlan Parquet partiționat (căutați PartitionFilters):')
partitioned_filter.explain('formatted')

Anul de filtrare folosit în toate interogările: 2002
Directoare Parquet partiționate (primele 20):
['Year=2001', 'Year=2002', 'Year=2003', 'Year=2004', 'Year=2005', 'Year=2006', 'Year=2007', 'Year=2008', 'Year=2009', 'Year=2010', 'Year=2011', 'Year=2012', 'Year=2013', 'Year=2014', 'Year=2015', 'Year=2016', 'Year=2017', 'Year=2018', 'Year=2019', 'Year=2020']
Fiecare cale Year=valoare corespunde valorii stocate în coloana Year.

Plan Parquet nepartiționat:
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [22]: [ID#1737L, Case Number#1738, Date#1739, Block#1740, IUCR#1741, Primary Type#1742, Description#1743, Location Description#1744, Arrest#1745, Domestic#1746, Beat#1747, District#1748, Ward#1749, Community Area#1750, FBI Code#1751, X Coordinate#1752, Y Coordinate#1753, Year#1754, Updated On#1755, Latitude#1756, Longitude#1757, Location#1758]
Batched: true
Location: InMemoryFileIndex [file:/content/drive/MyDrive/Colab Notebooks/P

### Dimensiuni și timpi

Pentru fiecare format se măsoară citirea cu `count()`, filtrarea cu `count()` și filtrarea urmată de agregarea după tipul infracțiunii cu `collect()`. Fiecare operație pornește de la o citire proaspătă, se repetă de trei ori și este raportată mediana; astfel Spark execută efectiv planul leneș.

In [25]:
loaders = {
    'CSV sursă': lambda: read_source_csv(file_path),
    'Parquet': lambda: spark.read.parquet(PARQUET_PATH),
    'Parquet partiționat': lambda: spark.read.parquet(PARTITIONED_PATH),
    'Delta partiționat': lambda: spark.read.format('delta').load(DELTA_PATH),
}

benchmark_results = {}
for format_name, loader in loaders.items():
    read_runs, read_median = measure(lambda: loader().count())
    filter_runs, filter_median = measure(
        lambda: loader().filter(col(PARTITION_COLUMN) == TEST_YEAR).count()
    )
    aggregation_runs, aggregation_median = measure(
        lambda: loader()
        .filter(col(PARTITION_COLUMN) == TEST_YEAR)
        .groupBy('Primary Type').count().collect()
    )
    benchmark_results[format_name] = {
        'read_runs': read_runs,
        'read_median': read_median,
        'filter_runs': filter_runs,
        'filter_median': filter_median,
        'aggregation_runs': aggregation_runs,
        'aggregation_median': aggregation_median,
    }
    print(
        f'{format_name}: citire={read_median:.2f}s; '
        f'filtrare={filter_median:.2f}s; agregare={aggregation_median:.2f}s'
    )

source_size_bytes = directory_size_bytes(RAW_DIR)
parquet_size_bytes = directory_size_bytes(PARQUET_PATH)
partitioned_size_bytes = directory_size_bytes(PARTITIONED_PATH)
delta_size_bytes = directory_size_bytes(DELTA_PATH)
parquet_saving_mb = (source_size_bytes - parquet_size_bytes) / (1024 ** 2)
parquet_saving_percent = 100 * (source_size_bytes - parquet_size_bytes) / source_size_bytes
print(f'CSV sursă: {size_in_mb(source_size_bytes):.2f} MB')
print(f'Parquet: {size_in_mb(parquet_size_bytes):.2f} MB')
print(f'Diferență CSV - Parquet: {parquet_saving_mb:.2f} MB ({parquet_saving_percent:.2f}%)')
print(f'Parquet partiționat: {size_in_mb(partitioned_size_bytes):.2f} MB')
print(f'Delta (include _delta_log): {size_in_mb(delta_size_bytes):.2f} MB')

CSV sursă: citire=6.55s; filtrare=18.14s; agregare=18.82s
Parquet: citire=0.90s; filtrare=0.61s; agregare=0.64s
Parquet partiționat: citire=1.10s; filtrare=0.37s; agregare=0.47s
Delta partiționat: citire=1.07s; filtrare=0.83s; agregare=0.69s
CSV sursă: 2291.91 MB
Parquet: 513.06 MB
Diferență CSV - Parquet: 1778.85 MB (77.61%)
Parquet partiționat: 522.06 MB
Delta (include _delta_log): 522.57 MB


### Inspectarea și versiunile Delta

Datele Delta sunt fișiere Parquet, iar `_delta_log` înregistrează tranzacțiile. După benchmark, aplicăm append, update și delete asupra unei înregistrări de laborator controlate, fără a schimba fișierele CSV sau Parquet.

In [27]:
delta_files = []
for root, _, filenames in os.walk(DELTA_PATH):
    for filename in filenames:
        relative = os.path.relpath(os.path.join(root, filename), DELTA_PATH)
        delta_files.append(relative)
parquet_file_count = len([name for name in delta_files if name.endswith('.parquet')])
delta_log_file_count = len([name for name in delta_files if name.startswith('_delta_log')])
print(f'Fișiere Parquet Delta: {parquet_file_count}')
print(f'Fișiere în _delta_log: {delta_log_file_count}')
delta_partition_directories = sorted({
    os.path.dirname(name) for name in delta_files
    if os.path.basename(os.path.dirname(name)).startswith(f'{PARTITION_COLUMN}=')
})
delta_log_files = sorted(name for name in delta_files if name.startswith('_delta_log'))
print('Directoare de partiție Delta (primele 20):', delta_partition_directories[:20])
print('Fișiere din _delta_log (primele 10):', delta_log_files[:10])
print('Fișiere Parquet din tabel:', [name for name in delta_files if name.endswith('.parquet')][:5])

delta_table = DeltaTable.forPath(spark, DELTA_PATH)
latest_version = delta_table.history(1).select('version').first()['version']
if latest_version != 0:
    raise RuntimeError('Tabelul Delta nu mai este la versiunea inițială 0. Rulați din nou celula de creare Delta.')

maximum_id = crimes_df.agg(max(col('ID')).alias('maximum_id')).first()['maximum_id']
if maximum_id is None:
    raise ValueError('Nu se poate crea înregistrarea de laborator: coloana ID este goală.')
LAB_RECORD_ID = int(maximum_id) + 1
lab_record = (
    crimes_df.filter(col(PARTITION_COLUMN).isNotNull()).limit(1)
    .withColumn('ID', lit(LAB_RECORD_ID).cast(LongType()))
    .withColumn('Case Number', lit('LAB-DELTA-APPEND'))
    .withColumn('Description', lit('LABORATORY APPEND'))
)
lab_record.write.format('delta').mode('append').save(DELTA_PATH)
assert spark.read.format('delta').load(DELTA_PATH).filter(col('ID') == LAB_RECORD_ID).count() == 1

delta_table = DeltaTable.forPath(spark, DELTA_PATH)
delta_table.update(
    condition=col('ID') == lit(LAB_RECORD_ID),
    set={'Description': lit('LABORATORY UPDATE')},
)
updated_record = spark.read.format('delta').load(DELTA_PATH).filter(col('ID') == LAB_RECORD_ID).first()
assert updated_record is not None and updated_record['Description'] == 'LABORATORY UPDATE'

delta_table = DeltaTable.forPath(spark, DELTA_PATH)
delta_table.delete(condition=col('ID') == lit(LAB_RECORD_ID))
assert spark.read.format('delta').load(DELTA_PATH).filter(col('ID') == LAB_RECORD_ID).count() == 0

delta_history = DeltaTable.forPath(spark, DELTA_PATH).history()
delta_history.select('version', 'operation', 'operationParameters').orderBy('version').show(truncate=False)
operations_by_version = {
    row['version']: row['operation']
    for row in delta_history.select('version', 'operation').collect()
}
assert operations_by_version.get(0) == 'WRITE'
assert operations_by_version.get(1) == 'WRITE'
assert operations_by_version.get(2) == 'UPDATE'
assert operations_by_version.get(3) == 'DELETE'
delta_v0 = spark.read.format('delta').option('versionAsOf', 0).load(DELTA_PATH)
delta_v2 = spark.read.format('delta').option('versionAsOf', 2).load(DELTA_PATH)
delta_current = spark.read.format('delta').load(DELTA_PATH)
assert delta_v0.count() == SOURCE_COUNT
assert delta_current.count() == SOURCE_COUNT
print(f'Rânduri în versiunea 0: {SOURCE_COUNT:,}; curent după delete: {SOURCE_COUNT:,}')
print('Înregistrarea de laborator în v0 (așteptat: fără rezultat):')
delta_v0.filter(col('ID') == LAB_RECORD_ID).select('ID', 'Case Number', 'Description').show(truncate=False)
print('Înregistrarea după update în v2:')
delta_v2.filter(col('ID') == LAB_RECORD_ID).select('ID', 'Case Number', 'Description').show(truncate=False)
print('Înregistrarea după delete în versiunea curentă (așteptat: fără rezultat):')
delta_current.filter(col('ID') == LAB_RECORD_ID).select('ID', 'Case Number', 'Description').show(truncate=False)

Fișiere Parquet Delta: 43
Fișiere în _delta_log: 4
Directoare de partiție Delta (primele 20): []
Fișiere din _delta_log (primele 10): ['_delta_log/.00000000000000000000.crc.crc', '_delta_log/.00000000000000000000.json.crc', '_delta_log/00000000000000000000.crc', '_delta_log/00000000000000000000.json']
Fișiere Parquet din tabel: ['col-6635a461-9cff-46e7-88f8-a1df0bc7dcaf=2022/part-00001-2ce93f53-544b-4132-8f44-0ba28130735d.c000.snappy.parquet', 'col-6635a461-9cff-46e7-88f8-a1df0bc7dcaf=2022/part-00002-edad06f8-0cbf-433e-89cf-ee43d0f6afbf.c000.snappy.parquet', 'col-6635a461-9cff-46e7-88f8-a1df0bc7dcaf=2024/part-00000-94c442ce-65cc-4339-8b3d-1bedde262b40.c000.snappy.parquet', 'col-6635a461-9cff-46e7-88f8-a1df0bc7dcaf=2024/part-00001-d13591b0-9da5-4f8a-81bf-f978772e1ab8.c000.snappy.parquet', 'col-6635a461-9cff-46e7-88f8-a1df0bc7dcaf=2023/part-00001-158c0d01-f7d4-4168-a696-1516743c2d65.c000.snappy.parquet']
+-------+---------+--------------------------------------------+
|version|operation|

In [29]:
import builtins

comparison_rows = []
format_sizes = {
    'CSV sursă': source_size_bytes,
    'Parquet': parquet_size_bytes,
    'Parquet partiționat': partitioned_size_bytes,
    'Delta partiționat': delta_size_bytes,
}
for format_name, results in benchmark_results.items():
    comparison_rows.append({
        'Format': format_name,
        'Dimensiune (MB)': builtins.round(size_in_mb(format_sizes[format_name]), 2),
        'Citire mediană (s)': builtins.round(results['read_median'], 2),
        'Filtrare mediană (s)': builtins.round(results['filter_median'], 2),
        'Agregare mediană (s)': builtins.round(results['aggregation_median'], 2),
        'Partiționat': format_name in ('Parquet partiționat', 'Delta partiționat'),
        'Istoric versiuni': format_name == 'Delta partiționat',
    })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

print(
    f'Concluzie măsurată: Parquet nepartiționat ocupă '
    f'{size_in_mb(parquet_size_bytes):.2f} MB față de '
    f'{size_in_mb(source_size_bytes):.2f} MB pentru CSV; '
    f'diferența este {parquet_saving_mb:.2f} MB '
    f'({parquet_saving_percent:.2f}%).'
)
print('Comparați medianele de citire, filtrare și agregare din tabel;')
print('PartitionFilters se verifică în planul de execuție, iar istoricul este disponibil numai în Delta Lake.')

,Format,Dimensiune (MB),Citire mediană (s),Filtrare mediană (s),Agregare mediană (s),Partiționat,Istoric versiuni
0,CSV sursă,2291.91,6.55,18.14,18.82,False,False
1,Parquet,513.06,0.90,0.61,0.64,False,False
2,Parquet partiționat,522.06,1.10,0.37,0.47,True,False
3,Delta partiționat,522.57,1.07,0.83,0.69,True,True


Concluzie măsurată: Parquet nepartiționat ocupă 513.06 MB față de 2291.91 MB pentru CSV; diferența este 1778.85 MB (77.61%).
Comparați medianele de citire, filtrare și agregare din tabel;
PartitionFilters se verifică în planul de execuție, iar istoricul este disponibil numai în Delta Lake.


## Concluzii

Concluzia comparativă se completează după rularea Etapei 4 în Colab. Tabelul generat raportează dimensiunea pe disc și medianele măsurate pentru CSV, Parquet, Parquet partiționat și Delta Lake. Interpretați rezultatele împreună cu `PartitionFilters` din planul formatted și istoricul Delta (versiunile 0–3); valorile depind de runtime și de accesul la Google Drive.

In [30]:
# Oprirea sesiunii Spark
spark.stop()
print("Sesiunea Spark a fost oprită.")

Sesiunea Spark a fost oprită.
